# Class Imbalance and Model Tuning

## 1. Objective

The baseline models showed that the class imbalance limited their ability to identify churned customers. The random forest produced the strongest baseline validation performance, with an F1 score of approximately 0.587 and an AUC-ROC of approximately 0.855, but its recall remained below 0.50.

This notebook will:

- Test class weighting and minority-class upsampling.
- Train logistic regression, decision tree, and random forest models.
- Tune promising model hyperparameters.
- Compare candidates using validation F1 and AUC-ROC.
- Select the final model configuration without using the test set.

The test set will remain untouched until final evaluation.

## 2. Imports and processed data

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.utils import resample, shuffle


In [ ]:
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

processed_dir = project_root / 'data' / 'processed'

In [ ]:
X_train = pd.read_csv(processed_dir / 'X_train.csv')
X_val = pd.read_csv(processed_dir / 'X_val.csv')

y_train = pd.read_csv(processed_dir / 'y_train.csv')['Exited']
y_val = pd.read_csv(processed_dir / 'y_val.csv')['Exited']

## 3. Verify the datasets

In [ ]:
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}')
print(f'y_val: {y_val.shape}')

print('\nTraining target proportions:')
print(y_train.value_counts(normalize=True).round(4))

print('\nValidation target proportions:')
print(y_val.value_counts(normalize=True).round(4))

In [ ]:
# Integrity checks

assert X_train.columns.equals(X_val.columns)
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_train.select_dtypes(include='object').empty
assert X_val.select_dtypes(include='object').empty

print('Training and validation datasets verified.')

## 4. Reproduce the strongest baseline

In [ ]:
# Create a reusable evaluation function

def evaluate_model(
    model_name,
    approach,
    model,
    X_fit,
    y_fit,
    X_evaluate,
    y_evaluate
):
    model.fit(X_fit, y_fit)

    predictions = model.predict(X_evaluate)
    probabilities = model.predict_proba(X_evaluate)[:, 1]

    results = {
        'Model': model_name,
        'Approach': approach,
        'Accuracy': accuracy_score(
            y_evaluate,
            predictions
        ),
        'Precision': precision_score(
            y_evaluate,
            predictions,
            zero_division=0
        ),
        'Recall': recall_score(
            y_evaluate,
            predictions,
            zero_division=0
        ),
        'F1': f1_score(
            y_evaluate,
            predictions,
            zero_division=0
        ),
        'AUC-ROC': roc_auc_score(
            y_evaluate,
            probabilities
        )
    }

    return model, results, predictions, probabilities

In [ ]:
# Recreate the unweighted random forest as a reference

baseline_forest = RandomForestClassifier(
    random_state=42
)

(
    baseline_forest,
    baseline_result,
    baseline_predictions,
    baseline_probabilities
) = evaluate_model(
    model_name='Random Forest',
    approach='Original imbalance',
    model=baseline_forest,
    X_fit=X_train,
    y_fit=y_train,
    X_evaluate=X_val,
    y_evaluate=y_val
)

pd.DataFrame([baseline_result]).round(4)

In [ ]:
# Initialize a list to collect all model experiments

experiment_results = [baseline_result]

## 5. Approach One: Class Weighting

Class weighting increases the penalty for misclassifying the minority class without changing the number of training observations. Using `class_weight='balanced'`, scikit-learn assigns weights inversely proportional to the class frequencies.

Only model training is affected. The validation set retains its original class distribution.

In [ ]:
# Create weighted models

weighted_models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000
    ),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    )
}

In [ ]:
# Train and evaluate models

weighted_results = []
weighted_outputs = {}

for model_name, model in weighted_models.items():
    (
        trained_model,
        result,
        predictions,
        probabilities
    ) = evaluate_model(
        model_name=model_name,
        approach='Class weighting',
        model=model,
        X_fit=X_train,
        y_fit=y_train,
        X_evaluate=X_val,
        y_evaluate=y_val
    )

    weighted_results.append(result)

    weighted_outputs[model_name] = {
        'model': trained_model,
        'predictions': predictions,
        'probabilities': probabilities
    }

In [ ]:
weighted_results_df = (
    pd.DataFrame(weighted_results)
    .sort_values(by='F1', ascending=False)
    .reset_index(drop=True)
)

weighted_results_df.round(4)

In [ ]:
# Compare with the original random forest

class_weight_comparison = pd.DataFrame(
    [baseline_result] + weighted_results
).sort_values(
    by='F1',
    ascending=False
).reset_index(drop=True)

class_weight_comparison.round(4)

In [ ]:
experiment_results = [baseline_result] + weighted_results

In [ ]:
# Confusion matrices

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, model_name in zip(axes, weighted_models):
    ConfusionMatrixDisplay.from_predictions(
        y_val,
        weighted_outputs[model_name]['predictions'],
        display_labels=['Retained', 'Churned'],
        cmap='Blues',
        ax=ax,
        colorbar=False
    )

    ax.set_title(f'{model_name}\nClass Weighting')

plt.tight_layout()
plt.show()

### Class-weighting findings

Class weighting increased recall for all three models, demonstrating that the models became more sensitive to churned customers.

The weighted logistic regression achieved the highest recall at approximately 0.663, but its precision decreased to approximately 0.376. This indicates that it identified more churned customers at the cost of producing considerably more false-positive churn predictions.

The weighted dicision tree improved upon the unrestricted baseline tree, achieving an F1 score of approximately 0.532 and an AUC-ROC of approximately 0.708. However, it remained weaker than the weighted random forest.

The weighted random forest produced the strongest class-weighted performance. Its validation F1 increased from approximately 0.587 to 0.631, primarily because recall improved fro approximately 0.467 to 0.622. Precision decreased from approximately 0.792 to 0.641, reflecting the tradeoff required to detect more churned customers.

Its AUC-ROC remained approximately unchanged at 0.856, suggesting that class weighting primarily changed the model's classification behavior rather than its overall ability to rank customers by churn risk.

The weighted random forest is the first candidate to exceed an F1 score of 0.59 on validation data. However, the final requirement must still be confirmed using the untouched test set.

## 6. Approach Two: Minority-Class Upsampling

Upsampling balances the training target by randomly duplicating observations from the minority churn class until it contains the same number of observations as the majority class.

Only the training data is resampled. The validation set retains its original class distribution so that model performance is evaluated under realistic conditions.

In [ ]:
# Separate the two training classes

majority_mask = y_train == 0
minority_mask = y_train == 1

X_train_majority = X_train[majority_mask]
y_train_majority = y_train[majority_mask]

X_train_minority = X_train[minority_mask]
y_train_minority = y_train[minority_mask]

print(f'Majority observations: {len(y_train_majority)}')
print(f'Minority observations: {len(y_train_minority)}')

In [ ]:
# Upsample the minority class

(
    X_train_minority_upsampled,
    y_train_minority_upsampled
) = resample(
    X_train_minority,
    y_train_minority,
    replace=True,
    n_samples=len(y_train_majority),
    random_state=42
)

In [ ]:
# Combine the majority and upsampled minority 

X_train_upsampled = pd.concat([
    X_train_majority,
    X_train_minority_upsampled  
])

y_train_upsampled = pd.concat([
    y_train_majority,
    y_train_minority_upsampled
])

In [ ]:
# Shuffle the upsampled dataset

X_train_upsampled, y_train_upsampled = shuffle(
    X_train_upsampled,
    y_train_upsampled,
    random_state=42
)

X_train_upsampled = X_train_upsampled.reset_index(drop=True)
y_train_upsampled = y_train_upsampled.reset_index(drop=True)

In [ ]:
# Verify the upsampled training data

print(f'Original training rows: {len(y_train)}')
print(f'Upsampled training rows: {len(y_train_upsampled)}')

print('\nUpsampled class counts:')
print(y_train_upsampled.value_counts())

print('\nUpsampled class proportions:')
print(y_train_upsampled.value_counts(normalize=True))

In [ ]:
assert y_train_upsampled.value_counts().nunique() == 1
assert len(X_train_upsampled) == len(y_train_upsampled)

print('Upsampled training data verified.')